In [16]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('MacOSX')
import matplotlib.pyplot as plt

# Generate simulated quarterly financial data for target companies
def generate_simulated_data(tickers, start_year, end_year):
    all_data = []
    for ticker in tickers:
        dates = pd.date_range(start=f"{start_year}-01-01", end=f"{end_year}-12-31", freq='Q')
        n = len(dates)        
        
        np.random.seed(42)
        base_assets = {"AAPL": 300000, "MSFT": 250000, "GOOGL": 200000}[ticker]
        base_liabilities = {"AAPL": 100000, "MSFT": 80000, "GOOGL": 70000}[ticker]
        base_revenue = {"AAPL": 80000, "MSFT": 60000, "GOOGL": 50000}[ticker]
        base_net_income = {"AAPL": 20000, "MSFT": 15000, "GOOGL": 12000}[ticker]
        shares_outstanding = {"AAPL": 16000, "MSFT": 7500, "GOOGL": 12000}[ticker]
        
        at = base_assets + np.linspace(0, 100000, n) + np.random.normal(0, 10000, n)
        lt = base_liabilities + np.linspace(0, 50000, n) + np.random.normal(0, 5000, n)
        revt = base_revenue + np.linspace(0, 30000, n) + np.random.normal(0, 3000, n)
        ni = base_net_income + np.linspace(0, 10000, n) + np.random.normal(0, 2000, n)
        
        df = pd.DataFrame({
            'tic': ticker,
            'datadate': dates,
            'at': at,
            'lt': lt,
            'revt': revt,
            'ni': ni,
            'csho': shares_outstanding
        })
        all_data.append(df)

    df = pd.concat(all_data).reset_index(drop=True)
    print(f"✅ Generated {len(df)} rows of simulated quarterly financial data.")
    return df

In [17]:
# Clean data and calculate core financial ratios
def process_data(df):
    df = df.dropna(subset=["at", "lt", "revt", "ni", "csho"])
    
    df["debt_ratio"] = df["lt"] / df["at"]
    df["roa"] = df["ni"] / df["at"]
    df["roe"] = df["ni"] / (df["at"] - df["lt"])
    df["eps"] = df["ni"] / df["csho"]
    
    df["datadate"] = pd.to_datetime(df["datadate"])
    df = df.sort_values(by=["tic", "datadate"]).reset_index(drop=True)
    print("✅ Data cleaning and ratio calculation completed.")
    return df

In [18]:
# Generate financial analysis report for single company
def generate_report(ticker, df):
    latest = df[df["tic"] == ticker].iloc[-1]
    report = f"""
==================== {ticker} Financial Analysis Report ====================
Report Date: {latest['datadate'].date()}

1. Scale & Solvency
- Total Assets: {latest['at']:,.2f}
- Total Liabilities: {latest['lt']:,.2f}
- Debt-to-Asset Ratio: {latest['debt_ratio']:.2%}
Status: {'High' if latest['debt_ratio'] > 0.6 else 'Moderate' if latest['debt_ratio'] > 0.3 else 'Low'}

2. Profitability
- Operating Revenue: {latest['revt']:,.2f}
- Net Income: {latest['ni']:,.2f}
- ROA: {latest['roa']:.2%}
- ROE: {latest['roe']:.2%}
- EPS: {latest['eps']:.2f}
Profitability: {'Excellent' if latest['roa'] > 0.05 else 'Average' if latest['roa'] > 0 else 'Weak'}

3. Brief Evaluation
The company's financial leverage is {'high' if latest['debt_ratio']>0.6 else 'moderate'},
profitability is {'good' if latest['roa']>0.03 else 'average'},
and overall financial health is {'healthy' if latest['roa']>0 and latest['debt_ratio']<0.7 else 'needs attention'}.
============================================================================
"""
    return report

In [19]:
# Plot and save ROA trend chart (macOS compatible)
def plot_roa_trend(df, save_path="roa_trend.png"):
    tickers = df["tic"].unique()
    plt.figure(figsize=(12, 5))
    
    for tic in tickers:
        company = df[df["tic"] == tic]
        plt.plot(company["datadate"], company["roa"], marker="o", label=tic)
    
    plt.title("ROA Trend (2019-2024)")
    plt.xlabel("Date")
    plt.ylabel("ROA")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()
    print(f"✅ ROA trend chart saved to {save_path}")

In [20]:
# Main program execution
if __name__ == "__main__":
    tickers = ["AAPL", "MSFT", "GOOGL"]
    start_year = 2019
    end_year = 2024

    raw_data = generate_simulated_data(tickers, start_year, end_year)
    data = process_data(raw_data)

    print("\nData Preview:")
    print(data.head())

    print("\n===== Financial Analysis Reports =====")
    for tic in tickers:
        report = generate_report(tic, data)
        print(report)

    plot_roa_trend(data)

/var/folders/6h/q3hnwl3910z7pkvbzxmsgc9c0000gn/T/ipykernel_38382/2922741319.py:11: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  dates = pd.date_range(start=f"{start_year}-01-01", end=f"{end_year}-12-31", freq='Q')


✅ Generated 72 rows of simulated quarterly financial data.
✅ Data cleaning and ratio calculation completed.

Data Preview:
    tic   datadate             at             lt          revt            ni  \
0  AAPL 2019-03-31  304967.141530   97278.086377  81030.854869  19928.347922   
1  AAPL 2019-06-30  302965.183075  102728.525992  76015.227360  23564.069920   
2  AAPL 2019-09-30  315172.537555   98592.858200  83580.947560  15630.075009   
3  AAPL 2019-12-31  328273.776825  108400.229222  82757.796637  22948.152835   
4  AAPL 2020-03-31  315049.770601  105692.458724  83186.625303  21913.224571   

    csho  debt_ratio       roa       roe       eps  
0  16000    0.318979  0.065346  0.095953  1.245522  
1  16000    0.339077  0.077778  0.117681  1.472754  
2  16000    0.312822  0.049592  0.072168  0.976880  
3  16000    0.330213  0.069906  0.104370  1.434260  
4  16000    0.335479  0.069555  0.104669  1.369577  

===== Financial Analysis Reports =====

==================== AAPL Financial A